# MCP Leaderboard Dashboard

Tracking adoption metrics for Payment, Commerce, and Crypto MCP servers.

**Data Sources:**
- npm Registry API (weekly downloads, trends)
- GitHub REST API (stars, forks, issues)

---

In [ ]:
import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import time

## MCP Registry

The curated list of Payment, Commerce, and Crypto MCP servers being tracked.

In [ ]:
# MCP Registry - VERIFIED list from reputable companies
# Last verified: 2026-02-23
# All npm packages and GitHub repos have been manually verified to exist

mcp_registry = [
    # ═══════════════════════════════════════════════════════════════════════════
    # CORE / INFRASTRUCTURE (Official MCP tooling)
    # ═══════════════════════════════════════════════════════════════════════════
    {"id": "mcp-sdk", "name": "MCP SDK", "category": "core", "npm_package": "@modelcontextprotocol/sdk", "github_repo": None, "description": "Official Model Context Protocol SDK"},
    {"id": "claude-code", "name": "Claude Code", "category": "core", "npm_package": "@anthropic-ai/claude-code", "github_repo": None, "description": "Anthropic Claude Code CLI"},
    {"id": "anthropic-sdk", "name": "Anthropic SDK", "category": "core", "npm_package": "@anthropic-ai/sdk", "github_repo": None, "description": "Official Anthropic API SDK"},
    {"id": "fastmcp", "name": "FastMCP", "category": "core", "npm_package": "fastmcp", "github_repo": "PrefectHQ/fastmcp", "description": "Fast, Pythonic MCP server framework"},
    {"id": "mcp-use", "name": "MCP Use", "category": "core", "npm_package": "mcp-use", "github_repo": "mcp-use/mcp-use", "description": "Fullstack MCP framework"},

    # ═══════════════════════════════════════════════════════════════════════════
    # DEVELOPER TOOLS (Code, browser, design, monitoring)
    # ═══════════════════════════════════════════════════════════════════════════
    {"id": "playwright-mcp", "name": "Playwright MCP", "category": "devtools", "npm_package": "@playwright/mcp", "github_repo": "microsoft/playwright-mcp", "description": "Microsoft Playwright browser automation"},
    {"id": "github-mcp", "name": "GitHub MCP", "category": "devtools", "npm_package": None, "github_repo": "github/github-mcp-server", "description": "GitHub's official MCP server"},
    {"id": "context7", "name": "Context7 (Upstash)", "category": "devtools", "npm_package": "@upstash/context7-mcp", "github_repo": "upstash/context7", "description": "Up-to-date code docs for LLMs"},
    {"id": "figma-mcp", "name": "Figma Context MCP", "category": "devtools", "npm_package": "figma-mcp", "github_repo": "GLips/Figma-Context-MCP", "description": "Figma layout info for AI agents"},
    {"id": "browserbase-mcp", "name": "Browserbase MCP", "category": "devtools", "npm_package": "@browserbasehq/mcp-server-browserbase", "github_repo": "browserbase/mcp-server-browserbase", "description": "Browser control with Stagehand"},
    {"id": "sentry-mcp", "name": "Sentry MCP", "category": "devtools", "npm_package": None, "github_repo": "getsentry/sentry-mcp", "description": "Sentry error monitoring integration"},

    # ═══════════════════════════════════════════════════════════════════════════
    # CLOUD / INFRASTRUCTURE
    # ═══════════════════════════════════════════════════════════════════════════
    {"id": "cloudflare-mcp", "name": "Cloudflare MCP", "category": "cloud", "npm_package": "@cloudflare/mcp-server-cloudflare", "github_repo": "cloudflare/mcp-server-cloudflare", "description": "Cloudflare API integration"},

    # ═══════════════════════════════════════════════════════════════════════════
    # PAYMENTS
    # ═══════════════════════════════════════════════════════════════════════════
    {"id": "stripe-mcp", "name": "Stripe MCP", "category": "payments", "npm_package": "@stripe/mcp", "github_repo": None, "description": "Official Stripe MCP server"},
    {"id": "paypal-agent-toolkit", "name": "PayPal Agent Toolkit", "category": "payments", "npm_package": "@paypal/agent-toolkit", "github_repo": "paypal/agent-toolkit", "description": "PayPal agent toolkit"},

    # ═══════════════════════════════════════════════════════════════════════════
    # COMMERCE
    # ═══════════════════════════════════════════════════════════════════════════
    {"id": "shopify-dev-mcp", "name": "Shopify Dev MCP", "category": "commerce", "npm_package": "@shopify/dev-mcp", "github_repo": None, "description": "Official Shopify developer MCP"},
    {"id": "shopify-mcp", "name": "Shopify MCP", "category": "commerce", "npm_package": "shopify-mcp", "github_repo": "GeLi2001/shopify-mcp", "description": "Community Shopify MCP (132 stars)"},
    {"id": "shopify-storefront-mcp", "name": "Shopify Storefront MCP", "category": "commerce", "npm_package": "@wolfielabs/shopify-storefront-mcp-server", "github_repo": None, "description": "Shopify storefront integration"},
    {"id": "shopify-mcp-server", "name": "Shopify MCP Server", "category": "commerce", "npm_package": "shopify-mcp-server", "github_repo": "amir-bengherbi/shopify-mcp-server", "description": "Community Shopify server"},

    # ═══════════════════════════════════════════════════════════════════════════
    # CRYPTO / WEB3
    # ═══════════════════════════════════════════════════════════════════════════
    {"id": "x402-mcp", "name": "x402 MCP", "category": "crypto", "npm_package": "x402-mcp", "github_repo": None, "description": "x402 protocol integration"},
    {"id": "mcpay", "name": "MCPay", "category": "crypto", "npm_package": "mcpay", "github_repo": "microchipgnu/MCPay", "description": "x402 payment infrastructure"},
    {"id": "near-mcp", "name": "NEAR MCP", "category": "crypto", "npm_package": None, "github_repo": "nearai/near-mcp", "description": "NEAR blockchain integration"},
]

df_registry = pd.DataFrame(mcp_registry)
print(f"Tracking {len(df_registry)} VERIFIED MCP servers across {df_registry['category'].nunique()} categories")
print()
df_registry.groupby('category').size().sort_values(ascending=False)

## Data Fetching Functions

Functions to pull live data from npm and GitHub APIs.

In [ ]:
def get_npm_weekly_downloads(package_name):
    """Fetch weekly downloads from npm registry API."""
    if package_name is None:
        return None
    try:
        # Use urllib.parse for more reliable encoding
        from urllib.parse import quote
        encoded = quote(package_name, safe='')
        url = f"https://api.npmjs.org/downloads/point/last-week/{encoded}"
        response = requests.get(url, timeout=15)
        if response.status_code == 200:
            data = response.json()
            return data.get('downloads', 0)
        else:
            print(f"  [npm WARN] {package_name}: HTTP {response.status_code}")
            return None
    except Exception as e:
        print(f"  [npm ERROR] {package_name}: {e}")
        return None


def get_npm_trend(package_name):
    """Fetch last 7 days of downloads for trend visualization."""
    if package_name is None:
        return []
    try:
        from urllib.parse import quote
        encoded = quote(package_name, safe='')
        url = f"https://api.npmjs.org/downloads/range/last-month/{encoded}"
        response = requests.get(url, timeout=15)
        if response.status_code == 200:
            data = response.json()
            downloads = data.get('downloads', [])
            return [d['downloads'] for d in downloads[-7:]]
        return []
    except Exception as e:
        print(f"  [trend ERROR] {package_name}: {e}")
        return []


def get_npm_change_percent(package_name):
    """Calculate week-over-week download change percentage."""
    if package_name is None:
        return 0
    try:
        from urllib.parse import quote
        encoded = quote(package_name, safe='')
        url = f"https://api.npmjs.org/downloads/range/last-month/{encoded}"
        response = requests.get(url, timeout=15)
        if response.status_code == 200:
            downloads = response.json().get('downloads', [])
            if len(downloads) < 14:
                return 0
            last_week = sum(d['downloads'] for d in downloads[-7:])
            prev_week = sum(d['downloads'] for d in downloads[-14:-7])
            if prev_week == 0:
                return 100 if last_week > 0 else 0
            return round(((last_week - prev_week) / prev_week) * 100)
        return 0
    except:
        return 0


def get_github_metrics(repo_path):
    """Fetch GitHub repository metrics."""
    if repo_path is None:
        return {'stars': 0, 'forks': 0, 'open_issues': 0, 'last_commit': None}
    try:
        url = f"https://api.github.com/repos/{repo_path}"
        headers = {'Accept': 'application/vnd.github.v3+json'}
        # Uncomment and add your token for higher rate limits:
        # headers['Authorization'] = 'token YOUR_GITHUB_TOKEN'
        response = requests.get(url, headers=headers, timeout=15)
        if response.status_code == 200:
            data = response.json()
            return {
                'stars': data.get('stargazers_count', 0),
                'forks': data.get('forks_count', 0),
                'open_issues': data.get('open_issues_count', 0),
                'last_commit': data.get('pushed_at')
            }
        elif response.status_code == 403:
            print(f"  [GitHub RATE LIMITED] {repo_path}")
        else:
            print(f"  [GitHub WARN] {repo_path}: HTTP {response.status_code}")
        return {'stars': 0, 'forks': 0, 'open_issues': 0, 'last_commit': None}
    except Exception as e:
        print(f"  [GitHub ERROR] {repo_path}: {e}")
        return {'stars': 0, 'forks': 0, 'open_issues': 0, 'last_commit': None}

## Fetch All MCP Data

Pull live metrics for all tracked MCPs. This may take a moment due to API rate limits.

In [ ]:
# Fetch data for all MCPs
mcp_data = []

print(f"Fetching metrics for {len(mcp_registry)} MCPs...")
print("=" * 60)

for idx, mcp in enumerate(mcp_registry):
    print(f"[{idx+1}/{len(mcp_registry)}] {mcp['name']}")
    
    # npm metrics
    weekly_downloads = get_npm_weekly_downloads(mcp['npm_package'])
    downloads_change = get_npm_change_percent(mcp['npm_package'])
    trend = get_npm_trend(mcp['npm_package'])
    
    # GitHub metrics
    gh = get_github_metrics(mcp['github_repo'])
    
    # Debug output
    dl_str = f"{weekly_downloads:,}" if weekly_downloads else "N/A"
    stars_str = f"{gh['stars']:,}" if gh['stars'] else "N/A"
    print(f"    npm: {dl_str} downloads | GitHub: {stars_str} stars")
    
    mcp_data.append({
        'id': mcp['id'],
        'name': mcp['name'],
        'category': mcp['category'],
        'description': mcp['description'],
        'npm_package': mcp['npm_package'],
        'github_repo': mcp['github_repo'],
        'weekly_downloads': weekly_downloads or 0,
        'downloads_change_pct': downloads_change,
        'trend': trend,
        'stars': gh['stars'],
        'forks': gh['forks'],
        'open_issues': gh['open_issues'],
        'last_commit': gh['last_commit']
    })
    
    time.sleep(0.5)  # Increased delay to avoid rate limits

df_mcps = pd.DataFrame(mcp_data)

# Rank by weekly downloads
df_mcps['rank'] = df_mcps['weekly_downloads'].rank(ascending=False, method='min').astype(int)
df_mcps = df_mcps.sort_values('rank')

print("=" * 60)
print(f"Completed! Fetched data for {len(df_mcps)} MCPs")
print(f"Total weekly downloads: {df_mcps['weekly_downloads'].sum():,}")
print(f"Total GitHub stars: {df_mcps['stars'].sum():,}")

## Summary Metrics

In [ ]:
# Calculate summary statistics
total_downloads = df_mcps['weekly_downloads'].sum()
total_stars = df_mcps['stars'].sum()
avg_change = df_mcps['downloads_change_pct'].mean()

print(f"{'='*50}")
print(f"MCP LEADERBOARD SUMMARY")
print(f"{'='*50}")
print(f"Total Weekly Downloads: {total_downloads:,}")
print(f"Total GitHub Stars:     {total_stars:,}")
print(f"Avg WoW Change:         {avg_change:+.1f}%")
print(f"MCPs Tracked:           {len(df_mcps)}")
print(f"{'='*50}")

## Leaderboard Table

In [ ]:
# Display leaderboard
df_leaderboard = df_mcps[['rank', 'name', 'category', 'weekly_downloads', 'downloads_change_pct', 'stars', 'forks']].copy()
df_leaderboard.columns = ['Rank', 'Name', 'Category', 'Weekly Downloads', 'WoW Change %', 'Stars', 'Forks']
df_leaderboard

## Downloads by Category

In [ ]:
# Downloads by category
df_category = df_mcps.groupby('category').agg({
    'weekly_downloads': 'sum',
    'stars': 'sum',
    'id': 'count'
}).rename(columns={'id': 'mcp_count'}).reset_index()

# Extended color palette for all categories
category_colors = {
    'core': '#8b5cf6',       # Purple - Core/Infrastructure
    'devtools': '#3b82f6',   # Blue - Developer Tools
    'cloud': '#06b6d4',      # Cyan - Cloud
    'payments': '#6366f1',   # Indigo - Payments
    'commerce': '#22c55e',   # Green - Commerce
    'crypto': '#f59e0b'      # Amber - Crypto
}

fig = px.bar(
    df_category,
    x='category',
    y='weekly_downloads',
    color='category',
    color_discrete_map=category_colors,
    title='Weekly Downloads by Category',
    labels={'weekly_downloads': 'Weekly Downloads', 'category': 'Category'},
    text='weekly_downloads'
)
fig.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
fig.update_layout(
    showlegend=False,
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    xaxis_title='',
    yaxis_title='Weekly Downloads'
)
fig.show()

## Top MCPs by Downloads

In [ ]:
# Top 10 MCPs by downloads
df_top10 = df_mcps.nlargest(10, 'weekly_downloads')

fig = px.bar(
    df_top10,
    x='weekly_downloads',
    y='name',
    color='category',
    color_discrete_map=category_colors,
    orientation='h',
    title='Top 10 MCPs by Weekly Downloads',
    labels={'weekly_downloads': 'Weekly Downloads', 'name': ''},
    text='weekly_downloads'
)
fig.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    legend_title='Category',
    xaxis_range=[0, df_top10['weekly_downloads'].max() * 1.15]  # Add space for labels
)
fig.show()

## GitHub Stars vs Downloads

In [ ]:
# Scatter plot: Stars vs Downloads
df_scatter = df_mcps[df_mcps['stars'] > 0].copy()

fig = px.scatter(
    df_scatter,
    x='stars',
    y='weekly_downloads',
    color='category',
    color_discrete_map=category_colors,
    size='forks',
    size_max=40,
    hover_name='name',
    hover_data=['description', 'npm_package', 'github_repo'],
    title='GitHub Stars vs Weekly Downloads',
    labels={'stars': 'GitHub Stars', 'weekly_downloads': 'Weekly Downloads'}
)
fig.update_layout(
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    legend_title='Category'
)
fig.show()

## Week-over-Week Growth Leaders

In [ ]:
# WoW growth chart
df_growth = df_mcps[df_mcps['downloads_change_pct'] != 0].nlargest(10, 'downloads_change_pct')

fig = px.bar(
    df_growth,
    x='downloads_change_pct',
    y='name',
    color='category',
    color_discrete_map=category_colors,
    orientation='h',
    title='Top 10 MCPs by Week-over-Week Growth',
    labels={'downloads_change_pct': 'WoW Change %', 'name': ''},
    text='downloads_change_pct'
)
fig.update_traces(texttemplate='%{text:+.0f}%', textposition='outside')
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    legend_title='Category'
)
fig.show()

## 7-Day Download Trends

In [ ]:
# Sparkline trends for top MCPs with trend data
df_with_trends = df_mcps[df_mcps['trend'].apply(lambda x: len(x) > 0)].nlargest(6, 'weekly_downloads')

if len(df_with_trends) > 0:
    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=df_with_trends['name'].tolist(),
        vertical_spacing=0.15,
        horizontal_spacing=0.08
    )
    
    for idx, (_, row) in enumerate(df_with_trends.iterrows()):
        r = idx // 3 + 1
        c = idx % 3 + 1
        trend = row['trend']
        days = list(range(1, len(trend) + 1))
        
        # Determine color based on trend direction
        color = '#22c55e' if trend[-1] >= trend[0] else '#ef4444'
        
        fig.add_trace(
            go.Scatter(
                x=days,
                y=trend,
                mode='lines',
                line=dict(color=color, width=2),
                fill='tozeroy',
                fillcolor=color.replace(')', ', 0.2)').replace('rgb', 'rgba') if 'rgb' in color else f"{color}33",
                showlegend=False,
                hovertemplate='Day %{x}: %{y:,.0f} downloads<extra></extra>'
            ),
            row=r, col=c
        )
    
    fig.update_layout(
        title='7-Day Download Trends (Top MCPs)',
        height=400,
        plot_bgcolor='rgba(0,0,0,0)',
        paper_bgcolor='rgba(0,0,0,0)'
    )
    fig.update_xaxes(title_text='', showticklabels=False)
    fig.update_yaxes(title_text='', showticklabels=False)
    fig.show()
else:
    print("No trend data available")

## Category Distribution

In [ ]:
# Pie chart of category distribution by downloads
fig = px.pie(
    df_category,
    values='weekly_downloads',
    names='category',
    color='category',
    color_discrete_map=category_colors,
    title='Download Share by Category',
    hole=0.4
)
fig.update_traces(textposition='outside', textinfo='percent+label')
fig.update_layout(
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)'
)
fig.show()

## Full Data Export

In [ ]:
# Export full dataset
df_export = df_mcps.drop(columns=['trend']).copy()
df_export['last_commit'] = pd.to_datetime(df_export['last_commit']).dt.strftime('%Y-%m-%d')
df_export

---

## Hex Input Parameters

To add interactivity in Hex, create these input parameters:

| Parameter Name | Type | Description | Default Value |
|---------------|------|-------------|---------------|
| `selected_category` | Dropdown | Filter by category | "all" |

Options for dropdown: `all`, `core`, `devtools`, `cloud`, `payments`, `commerce`, `crypto`

Then modify the data loading cell to filter:
```python
if selected_category != 'all':
    df_mcps = df_mcps[df_mcps['category'] == selected_category]
```

## Data Sources

| Category | Companies |
|----------|-----------|
| **Core** | Anthropic, Model Context Protocol, Prefect |
| **DevTools** | Microsoft, GitHub, Upstash, Figma, Browserbase, Sentry |
| **Cloud** | Cloudflare |
| **Payments** | Stripe, PayPal |
| **Commerce** | Shopify |
| **Crypto** | x402, MCPay, NEAR |